# 22. Component test in two further architectures

Notebook 14 confirmed for EfficientNet-B0 that the repair from quality-aware augmentation comes from its JPEG component (`qaug_jpeg` closed 103% of the gap, `qaug_resize` 11%). This notebook repeats the test for MobileNetV3-Large and MobileNetV4-Conv-S so that every panel of the ablation figure has the same four arms: `base`, `qaug`, `qaug_jpeg`, `qaug_resize`.

Nothing changes from notebook 14 except the model name: data, split, training code, `rand_quality_parts`, the `degrade` protocol and seeds 42/1/2. `base` and `qaug` reuse the checkpoints of notebook 08 when found and are checked against the stored numbers; missing ones are retrained.

## Pre-specified criteria

Written on 20 November 2026 before any new arm was trained and recorded in `preregistration.json`. Per architecture, at **112 px + JPEG q70**, gap = other→disease of `base` − `qaug`:

- **G** three seeds per arm and a gap of at least **10** points. Notebook 14 used 20; Table 3 of the paper already showed a gap of about 11 points for MobileNetV4-Conv-S, so 20 would have made the test undecidable from the start. The threshold was set from existing `base`/`qaug` results only.
- **P1** `qaug_jpeg` closes at least 80% of the gap and the upper 95% CI of (qaug_jpeg − base) is below zero.
- **P2** `qaug_resize` closes less than 20%.
- Verdict per architecture: G, P1 and P2 → *confirmed*; G but not P1 or P2 → *not confirmed*; no G → *undecidable*.

Everything else (80 px, accuracy, disease→other, other levels) is descriptive.

## Running

Kaggle GPU (T4, Internet on). Inputs: the private dataset and the output of notebook 08 with the `base`/`qaug` checkpoints of both architectures. Results are written to `arch22_results.zip` (without `best.pt`). Twelve new runs take about 1.5–2.5 hours on a T4.

In [ ]:
# 0. Settings and pre-specified criteria
import os, sys, io, re, json, time, math, random, shutil, zipfile, subprocess, warnings, datetime
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

T_START = time.time()
IN_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
IN_COLAB = (not IN_KAGGLE) and Path('/var/colab/hostname').exists()
MOCK = os.environ.get('JPEG22_MOCK') == '1'      # synthetic data, for testing the code only

# ---------- Pre-specified criteria (20 Nov 2026). Do not edit after results are seen. ----------
CRIT = dict(
    level='112px',                   # 112 px + JPEG q70 (as in 14)
    G_min_gap=10.0, G_min_seeds=3,   # base - qaug gap >= 10 pts (14 used 20; Table 3 showed a gap of about 11 for MobileNetV4-Conv-S)
    P1_jpeg_close_min=80.0,          # qaug_jpeg closes >= 80% and the upper CI of (qaug_jpeg - base) is < 0
    P2_resize_close_max=20.0,        # qaug_resize closes < 20%
    architectures=['mobilenetv3_large_100.ra_in1k', 'mobilenetv4_conv_small.e2400_r224_in1k'],
)

MODELS = list(CRIT['architectures'])       # same timm names as 03/08
PRETRAINED = True
SEEDS = [42, 1, 2]
NEW_ARMS = ['qaug_jpeg', 'qaug_resize']
REF_ARMS = ['base', 'qaug']
ARMS = REF_ARMS + NEW_ARMS
ARM_SPEC = {                                # as in 14
    'base': None,
    'qaug': dict(parts=('resize', 'jpeg', 'noise'), who='all'),
    'qaug_jpeg': dict(parts=('jpeg',), who='all'),
    'qaug_resize': dict(parts=('resize',), who='all'),
}
EPOCHS, PATIENCE, BATCH = 30, 8, 64
LR, WD, LABEL_SMOOTH = 1e-3, 0.05, 0.1
LEVELS, JPEG_Q = [224, 160, 112, 80], 70
IMG_SIZE = 224
Q_P_DOWNSCALE, Q_SCALE, Q_P_JPEG, Q_JPEG, Q_P_NOISE = 0.6, (0.25, 1.0), 0.6, (30, 96), 0.3
REUSE_CKPT = True          # base/qaug: reuse best.pt from 08 when found
TRAIN_MISSING_REF = True   # retrain base/qaug if no checkpoint is found
SAVE_CKPT = True
N_BOOT = 2000
MAX_TRAIN_HOURS = 10.5

if MOCK:
    MODELS = ['mobilenetv3_small_050', 'mobilenetv3_small_075']
    PRETRAINED = False
    SEEDS, EPOCHS, PATIENCE, BATCH, N_BOOT = [42, 1], 2, 8, 16, 200
    CRIT = dict(CRIT, G_min_seeds=2)
print(f'platform: kaggle={IN_KAGGLE} colab={IN_COLAB} mock={MOCK} | models {MODELS}')


In [ ]:
# 1. Locate data (from 14), index existing checkpoints, record criteria
def _is_v11(p):
    s = str(p).lower()
    return ('v11' in s) or ('v1_1' in s)

def find_data_dir(root):
    c = [p.parent for p in Path(root).rglob('cache256.npy') if (p.parent / 'metadata.csv').exists()]
    if not c:
        raise FileNotFoundError(f'cache256.npy + metadata.csv not found under {root}')
    c.sort(key=lambda p: (_is_v11(p), str(p)))   # prefer v1
    return c[0]

if IN_KAGGLE:
    DATA_DIR = find_data_dir('/kaggle/input')
    OUT_DIR = Path('/kaggle/working/arch22')
    CACHE_PATH = DATA_DIR / 'cache256.npy'
    SEARCH_ROOTS = [Path('/kaggle/input')]
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/durian_work')
    OUT_DIR = DATA_DIR / 'results' / 'arch22'
    CACHE_PATH = Path('/content/cache256.npy')          # local copy; Drive reads drop out
    if not CACHE_PATH.exists():
        print('copying cache256.npy to local disk (1.2 GB)')
        shutil.copy(DATA_DIR / 'cache256.npy', CACHE_PATH)
    SEARCH_ROOTS = [DATA_DIR / 'results']
else:
    DATA_DIR = Path(os.environ.get('DURIAN_DATA_DIR', '.'))
    OUT_DIR = Path(os.environ.get('DURIAN_OUT_DIR', './arch22'))
    CACHE_PATH = DATA_DIR / 'cache256.npy'
    SEARCH_ROOTS = [DATA_DIR] + [Path(p) for p in os.environ.get('DURIAN_EXTRA_DIRS', '').split(os.pathsep) if p]
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_ROOTS = [r for r in SEARCH_ROOTS if Path(r).exists()]

# Record the criteria before anything else runs. If a record exists and differs, the criteria were edited: stop.
PREREG = OUT_DIR / 'preregistration.json'
_prereg = dict(criteria=CRIT, arms=ARM_SPEC, new_arms=NEW_ARMS, seeds=SEEDS, models=MODELS,
               quality_aug=dict(p_downscale=Q_P_DOWNSCALE, scale=Q_SCALE, p_jpeg=Q_P_JPEG, jpeg_q=Q_JPEG, p_noise=Q_P_NOISE))
if PREREG.exists():
    _old = json.loads(PREREG.read_text())
    assert json.loads(json.dumps(_prereg['criteria'])) == _old['criteria'], 'CRIT differs from the recorded preregistration.json; criteria must not change after the first run'
else:
    PREREG.write_text(json.dumps(dict(written_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(), **_prereg), indent=1, default=list))

meta_all = pd.read_csv(DATA_DIR / 'metadata.csv')
X = np.load(CACHE_PATH, mmap_mode='r')
assert len(meta_all) == X.shape[0], f'metadata has {len(meta_all)} rows but the cache has {X.shape[0]}'
assert X.shape[1:] == (256, 256, 3) and X.dtype == np.uint8, X.shape
meta_all['row'] = np.arange(len(meta_all))

# With the v1.1 dataset attached, orchard rows (group starting with orchard_) are kept out of training and used as a field test set.
is_orch = meta_all['group'].astype(str).str.startswith('orchard_') if 'group' in meta_all else pd.Series(False, index=meta_all.index)
meta = meta_all[~is_orch].reset_index(drop=True)
FIELD = {}                                               # set name -> uint8 N x 256 x 256 x 3 (non-durian orchard photographs)
if is_orch.any():
    FIELD['O1_A'] = np.stack([X[r] for r in meta_all.loc[is_orch, 'row']])
    print(f'{int(is_orch.sum())} orchard rows in metadata removed from training and kept as field set O1_A')
print('DATA_DIR =', DATA_DIR, '| OUT_DIR =', OUT_DIR, '| v1 images =', len(meta))
if not MOCK and len(meta) != 6051:
    print(f'WARNING: expected 6,051 v1 images, found {len(meta)}. Check the attached dataset.')

def load_classes():
    labels = sorted(meta['label'].unique())
    for p in sorted(Path(DATA_DIR).rglob('classes.json')):
        try:
            j = json.load(open(p, encoding='utf-8'))
            if isinstance(j, dict):
                j = j.get('classes', j)
            if isinstance(j, dict):
                if all(str(k).isdigit() for k in j):
                    j = [j[k] for k in sorted(j, key=int)]
                else:
                    j = [k for k, _ in sorted(j.items(), key=lambda kv: kv[1])]
            if sorted(j) == labels:
                return list(j)
        except Exception as e:
            print('could not read', p, ':', e)
    print('no matching classes.json; classes sorted alphabetically')
    return labels

CLASSES = load_classes()
C = len(CLASSES); OTHER = CLASSES.index('other')
meta['y'] = meta['label'].map({c: i for i, c in enumerate(CLASSES)}).astype(int)
meta['kind'] = np.where(meta['label'] == 'other', 'other', 'disease')
if 'source' not in meta: meta['source'] = meta['kind']
if 'group' not in meta: meta['group'] = meta['row'].astype(str)
SPLIT = {s: meta[meta['split'] == s] for s in ['train', 'val', 'test']}
IDX = {s: SPLIT[s]['row'].values for s in SPLIT}              # cache row indices (as IDX in 03/08)
y_all = np.full(len(meta_all), -1); y_all[meta['row'].values] = meta['y'].values
ROW_GROUP = dict(zip(meta['row'].values, meta['group'].astype(str).values))
print({s: len(d) for s, d in SPLIT.items()}, '| classes', C, '| other in train/test =',
      int((SPLIT['train'].kind == 'other').sum()), int((SPLIT['test'].kind == 'other').sum()))

# ---------- existing checkpoints (03/08 in the dataset or output of 08, earlier runs of 22) ----------
CKPT_INDEX = {}
for _root in SEARCH_ROOTS:
    for _p in sorted(Path(_root).rglob('best.pt')):
        if _is_v11(_p) or OUT_DIR in _p.parents: continue          # skip models trained on v1.1
        CKPT_INDEX.setdefault(_p.parent.name, []).append(_p.parent)

def mshort(model): return model.split('.')[0]

def find_ckpt(model, arm, seed):
    # best.pt folder for (model, arm, seed) trained on v1. 03/08 naming: <model>__<arm>__s<seed>; 22 naming: <model>/<arm>__s<seed>
    for name in [f'{mshort(model)}__{arm}__s{seed}', f'{arm}__s{seed}']:
        for d in CKPT_INDEX.get(name, []):
            j = {}
            if (d / 'result.json').exists():
                try: j = json.loads((d / 'result.json').read_text())
                except Exception: j = {}
            if 'train_n' in j and int(j['train_n']) != len(IDX['train']): continue
            if name == f'{arm}__s{seed}' and (d.parent.name != mshort(model) or j.get('model') not in (None, model)): continue
            if name == f'{arm}__s{seed}' and j.get('arm') not in (None, arm): continue
            return d
    return None

_miss = []
print('reusable checkpoints:')
for _m in MODELS:
    for _a in ARMS:
        for _s in SEEDS:
            _d = find_ckpt(_m, _a, _s)
            if _d is not None: print(f'   {mshort(_m)}/{_a}__s{_s:<3} ← {_d}')
            elif _a in REF_ARMS: _miss.append(f'{mshort(_m)}/{_a}__s{_s}')
if _miss:
    print('WARNING: reference checkpoints not found:', _miss)
    print('   -> ' + ('retrained below (base/qaug will differ slightly from Table 3; report this)' if TRAIN_MISSING_REF else 'TRAIN_MISSING_REF = False: these arms will have no results') +
          '. To reuse the original checkpoints, attach the output of notebook 08.')


In [ ]:
# 2. Degradation protocol (copied verbatim from 14)
def jpeg(im, q):
    b = io.BytesIO(); im.save(b, 'JPEG', quality=int(q)); b.seek(0)
    return Image.open(b).convert('RGB')

def degrade(a, s=160, q=70):                      # evaluation protocol, copied verbatim from 03/08
    im = Image.fromarray(a).resize((s, s), Image.BILINEAR)
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=q)
    return np.asarray(Image.open(buf).convert('RGB').resize((256, 256), Image.BICUBIC))

# standard levels (names as in 03/08), plus variants that separate JPEG from resizing (as in 13)
STD_LEVELS = ['original'] + [f'{L}px' for L in LEVELS]
VARIANTS = STD_LEVELS + ['jpeg_only'] + [f'{L}px_nojpeg' for L in LEVELS[1:]]

def apply_variant(a, name):
    a = np.asarray(a)
    if name == 'original': return a
    if name == 'jpeg_only': return np.asarray(jpeg(Image.fromarray(a), JPEG_Q))
    L = int(name.split('px')[0])
    if name.endswith('_nojpeg'):
        return np.asarray(Image.fromarray(a).resize((L, L), Image.BILINEAR).resize((256, 256), Image.BICUBIC))
    return degrade(a, L, JPEG_Q)

print('protocol ready | variants:', VARIANTS)


In [ ]:
# 3. Training setup (from 14; make_model takes the model name)
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False
DO_B = HAS_GPU or MOCK
if not DO_B:
    print('no GPU: training skipped (the summary cell still runs on existing results). Run on a Kaggle GPU.')

if DO_B:
    try:
        import timm
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'timm'], check=False); import timm
    import torchvision.transforms.v2 as T
    from torch.utils.data import Dataset, DataLoader
    from sklearn.metrics import accuracy_score, f1_score
    dev = 'cuda' if HAS_GPU else 'cpu'
    MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    NUM_WORKERS = 0 if MOCK else (4 if IN_KAGGLE else 2)
    XR = np.load(CACHE_PATH)                     # whole cache in RAM

    def rand_quality(a, rng):                                   # copied verbatim from 03/08/13 (used by the qaug arm)
        if rng.random() < Q_P_DOWNSCALE:
            s = max(16, int(256 * rng.uniform(*Q_SCALE)))
            im = Image.fromarray(a).resize((s, s), Image.BILINEAR)
            a = np.asarray(im.resize((256, 256), rng.choice([Image.BILINEAR, Image.BICUBIC, Image.NEAREST])))
        if rng.random() < Q_P_JPEG:
            buf = io.BytesIO(); Image.fromarray(a).save(buf, 'JPEG', quality=int(rng.integers(*Q_JPEG)))
            a = np.asarray(Image.open(buf).convert('RGB'))
        if rng.random() < Q_P_NOISE:
            a = np.clip(a.astype(np.float32) + rng.normal(0, rng.uniform(2, 8), a.shape), 0, 255).astype(np.uint8)
        return a

    def rand_quality_parts(a, rng, parts, p_jpeg=Q_P_JPEG):     # identical to rand_quality, plus a switch per component
        if 'resize' in parts and rng.random() < Q_P_DOWNSCALE:
            s = max(16, int(256 * rng.uniform(*Q_SCALE)))
            im = Image.fromarray(a).resize((s, s), Image.BILINEAR)
            a = np.asarray(im.resize((256, 256), rng.choice([Image.BILINEAR, Image.BICUBIC, Image.NEAREST])))
        if 'jpeg' in parts and rng.random() < p_jpeg:
            buf = io.BytesIO(); Image.fromarray(a).save(buf, 'JPEG', quality=int(rng.integers(*Q_JPEG)))
            a = np.asarray(Image.open(buf).convert('RGB'))
        if 'noise' in parts and rng.random() < Q_P_NOISE:
            a = np.clip(a.astype(np.float32) + rng.normal(0, rng.uniform(2, 8), a.shape), 0, 255).astype(np.uint8)
        return a

    # With all components on, the output must equal rand_quality pixel for pixel (same seed); with one on, only that change may occur.
    _a0 = np.clip(np.asarray(X[int(IDX['train'][0])]).astype(np.int16) + np.random.default_rng(0).integers(-6, 7, (256, 256, 3)), 0, 255).astype(np.uint8)   # add texture so the image is not flat
    for _s in range(40):
        assert np.array_equal(rand_quality(_a0, np.random.default_rng(_s)), rand_quality_parts(_a0, np.random.default_rng(_s), ('resize', 'jpeg', 'noise'))), _s
    assert all(np.array_equal(rand_quality_parts(_a0, np.random.default_rng(_s), ()), _a0) for _s in range(5))
    _n = [not np.array_equal(rand_quality_parts(_a0, np.random.default_rng(_s), ('jpeg',), 1.0), _a0) for _s in range(20)]
    assert all(_n), 'jpeg with p=1.0 must change every image'
    _fr = {p: np.mean([not np.array_equal(rand_quality_parts(_a0, np.random.default_rng(_s), (p,)), _a0) for _s in range(1000)]) for p in ('resize', 'jpeg', 'noise')}
    assert abs(_fr['resize'] - Q_P_DOWNSCALE) < 0.06 and abs(_fr['jpeg'] - Q_P_JPEG) < 0.06 and abs(_fr['noise'] - Q_P_NOISE) < 0.06, _fr
    print('rand_quality_parts matches rand_quality | share of images changed with one component on (expected 0.6/0.6/0.3):', {k: round(float(v), 2) for k, v in _fr.items()})

    tf_train = T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), antialias=True),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
        T.ColorJitter(0.35, 0.35, 0.25, 0.02),
        T.RandomApply([T.GaussianBlur(5, (0.1, 2.0))], p=0.2),
        T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])
    tf_eval = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
                         T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])

    def arm_applies(spec, y):
        return spec is not None and (spec['who'] == 'all' or (spec['who'] == 'other' and y == OTHER) or (spec['who'] == 'disease' and y != OTHER))

    class TrainDS(Dataset):
        def __init__(self, idx, arm):
            self.idx, self.arm, self.spec = np.asarray(idx), arm, ARM_SPEC[arm]
        def __len__(self): return len(self.idx)
        def __getitem__(self, k):
            i = int(self.idx[k]); y = int(y_all[i])
            rng = np.random.default_rng(torch.randint(0, 2**31 - 1, (1,)).item())
            a = XR[i]
            if self.arm == 'qaug': a = rand_quality(a, rng)                      # original path of 03/08/13
            elif arm_applies(self.spec, y): a = rand_quality_parts(a, rng, self.spec['parts'], self.spec.get('p_jpeg', Q_P_JPEG))
            return tf_train(torch.from_numpy(np.array(a)).permute(2, 0, 1)), y

    @torch.no_grad()
    def logits_of(model, arrs):
        model.eval(); out = []
        for s in range(0, len(arrs), BATCH):
            x = torch.stack([tf_eval(torch.from_numpy(np.array(a)).permute(2, 0, 1)) for a in arrs[s:s + BATCH]])
            with torch.autocast(dev, enabled=(dev == 'cuda')):
                out.append(model(x.to(dev, non_blocking=True)).float().cpu())
        return torch.cat(out)

    @torch.no_grad()
    def preds_multi(models, arrs):
        """Like logits_of, but preprocesses once for several models. Returns {name: predicted class}."""
        out = {t: [] for t in models}
        for s in range(0, len(arrs), BATCH):
            x = torch.stack([tf_eval(torch.from_numpy(np.array(a)).permute(2, 0, 1)) for a in arrs[s:s + BATCH]]).to(dev, non_blocking=True)
            with torch.autocast(dev, enabled=(dev == 'cuda')):
                for t, m in models.items(): out[t].append(m(x).float().argmax(1).cpu())
        return {t: torch.cat(v).numpy().astype(np.uint8) for t, v in out.items()}

    def ece(probs, y, n_bins=15):
        conf, pred = probs.max(1), probs.argmax(1); acc = (pred == y); e = 0.0
        edges = np.linspace(0, 1, n_bins + 1)
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (conf > lo) & (conf <= hi)
            if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
        return float(e)

    def fit_temperature(logits, y):
        logT = torch.zeros(1, requires_grad=True)
        opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=200); yt = torch.as_tensor(y)
        def closure():
            opt.zero_grad(); loss = torch.nn.functional.cross_entropy(logits / logT.exp(), yt); loss.backward(); return loss
        opt.step(closure)
        return float(logT.detach().exp())

    def metrics(p, y):
        pred = p.argmax(1); o = y == OTHER
        return dict(acc=100 * accuracy_score(y, pred), macro_f1=100 * f1_score(y, pred, average='macro'),
                    other_as_disease=100 * float((pred[o] != OTHER).mean()), disease_as_other=100 * float((pred[~o] == OTHER).mean()),
                    acc_disease=100 * float((pred[~o] == y[~o]).mean()), ece=100 * ece(p, y))

    def make_model(model_name):
        return timm.create_model(model_name, pretrained=PRETRAINED, num_classes=C, drop_rate=0.2).to(dev)

    if DO_B:
        t0 = time.time()
        TEST_ARRS = {v: [apply_variant(XR[i], v) for i in IDX['test']] for v in VARIANTS}
        VAL_ARRS = [XR[i] for i in IDX['val']]; VAL_DEG = [degrade(XR[i], 112) for i in IDX['val']]
        print(f'{len(VARIANTS)} evaluation variants prepared ({time.time()-t0:.0f}s) | device {dev}')

In [ ]:
# 4. Train and evaluate (from 14; takes the model name and writes one folder per model)
def evaluate_and_save(model, out, arm, seed, info, model_name):
    yt = y_all[IDX['test']]
    T_ = fit_temperature(logits_of(model, VAL_ARRS), y_all[IDX['val']])
    rob, probs = [], {}
    for v in VARIANTS:
        p = (logits_of(model, TEST_ARRS[v]) / T_).softmax(1).numpy(); probs[v] = p.astype(np.float32)
        rob.append(dict(variant=v, **metrics(p, yt)))
    pd.DataFrame(rob).to_csv(out / 'robustness.csv', index=False)
    np.savez_compressed(out / 'test_probs.npz', idx=IDX['test'], y=yt, **probs)
    (out / 'result.json').write_text(json.dumps(dict(arm=arm, seed=seed, model=model_name, temperature=round(T_, 3), train_n=int(len(IDX['train'])),
        torch_version=torch.__version__, timm_version=timm.__version__, gpu=(torch.cuda.get_device_name(0) if HAS_GPU else 'cpu'), robustness=rob, **info), indent=1))
    r = {d['variant']: d for d in rob}
    print(f'{out.name} [{info["source"]}]: acc {r["original"]["acc"]:.2f} | other->disease original/112/80 = '
          f'{r["original"]["other_as_disease"]:.1f}/{r["112px"]["other_as_disease"]:.1f}/{r["80px"]["other_as_disease"]:.1f}')
    return r

def old_numbers(ref):
    """Read a stored robustness.csv -> {level: (acc %, other->disease %)}. 03/08 format (column level, fractions) or 13/14 (column variant, %)."""
    f = ref / 'robustness.csv'
    if not f.exists(): return {}
    d = pd.read_csv(f)
    if 'level' in d.columns: return {str(r['level']): (100 * r['acc'], 100 * r['other_as_disease']) for _, r in d.iterrows()}
    if 'variant' in d.columns: return {str(r['variant']): (r['acc'], r['other_as_disease']) for _, r in d.iterrows()}
    return {}

def reuse_checkpoint(model_name, arm, seed, out):
    # Models trained earlier: evaluate the stored best.pt with this code and check that it reproduces the stored numbers.
    if not REUSE_CKPT: return False
    ref = find_ckpt(model_name, arm, seed)
    if ref is None: return False
    model = make_model(model_name); model.load_state_dict(torch.load(ref / 'best.pt', map_location=dev))
    r = evaluate_and_save(model, out, arm, seed, dict(source='reused_checkpoint', ref_dir=str(ref)), model_name)
    old = old_numbers(ref); lv = [l for l in STD_LEVELS if l in old]
    j = json.loads((out / 'result.json').read_text())
    if lv:
        d_acc = max(abs(r[l]['acc'] - old[l][0]) for l in lv); d_oad = max(abs(r[l]['other_as_disease'] - old[l][1]) for l in lv)
        ok = d_acc <= 0.35 and d_oad <= 2.7           # about 3 of 888 images / 2 of 77 (GPU and AMP nondeterminism)
        print(f'   check against stored results ({ref}): max diff acc {d_acc:.2f} pts, other->disease {d_oad:.2f} pts -> ' + ('match' if ok else 'MISMATCH: evaluation differs from training run, stop here'))
        j['ref_check'] = dict(max_abs_acc=d_acc, max_abs_other_as_disease=d_oad, ok=bool(ok))
    else:
        print(f'   WARNING: no stored robustness.csv in {ref} to check against'); j['ref_check'] = dict(ok=None)
    (out / 'result.json').write_text(json.dumps(j, indent=1))
    return True

def train_one(model_name, arm, seed):
    out = OUT_DIR / mshort(model_name) / f'{arm}__s{seed}'
    if (out / 'result.json').exists():
        print(f'skip {out.name} (already done)'); return
    out.mkdir(parents=True, exist_ok=True)
    if reuse_checkpoint(model_name, arm, seed, out): return
    if arm in REF_ARMS and not TRAIN_MISSING_REF:
        print(f'skip {out}: no reference checkpoint and TRAIN_MISSING_REF = False'); out.rmdir(); return
    if (time.time() - T_START) / 3600 > MAX_TRAIN_HOURS:
        print(f'over {MAX_TRAIN_HOURS} h: {out.name} not started (rerun with this output attached as input to train only what is missing)'); out.rmdir(); return
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    tr = IDX['train']
    counts = np.bincount(y_all[tr], minlength=C)
    w = torch.tensor(len(tr) / (C * counts), dtype=torch.float32, device=dev)
    crit = torch.nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTH)
    model = make_model(model_name)
    dl = DataLoader(TrainDS(tr, arm), batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS,
                    pin_memory=(dev == 'cuda'), drop_last=True, persistent_workers=NUM_WORKERS > 0)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl), pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=(dev == 'cuda'))
    best, best_ep, hist, t0 = -1, 0, [], time.time()
    yv = y_all[IDX['val']]
    for ep in range(EPOCHS):
        model.train(); tot = 0
        for x, yb in dl:
            x, yb = x.to(dev, non_blocking=True), yb.to(dev, non_blocking=True)
            with torch.autocast(dev, enabled=(dev == 'cuda')):
                loss = crit(model(x), yb)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(yb)
        f_clean = f1_score(yv, logits_of(model, VAL_ARRS).argmax(1).numpy(), average='macro')
        f_deg = f1_score(yv, logits_of(model, VAL_DEG).argmax(1).numpy(), average='macro')
        score = (f_clean + f_deg) / 2
        hist.append(dict(epoch=ep + 1, train_loss=tot / len(dl.dataset), val_f1_clean=f_clean, val_f1_112px=f_deg))
        print(f'  [{mshort(model_name)} {arm} s{seed}] ep {ep + 1:2d} | loss {hist[-1]["train_loss"]:.3f} | val F1 clean {f_clean:.4f} | 112px {f_deg:.4f}')
        if score > best:
            best, best_ep = score, ep + 1; torch.save(model.state_dict(), out / 'best.pt')
        elif ep + 1 - best_ep >= PATIENCE:
            print('  early stop'); break
    del dl
    model.load_state_dict(torch.load(out / 'best.pt', map_location=dev))
    pd.DataFrame(hist).to_csv(out / 'history.csv', index=False)
    evaluate_and_save(model, out, arm, seed, dict(source='trained', best_epoch=best_ep, train_minutes=round((time.time() - t0) / 60, 1)), model_name)
    if not SAVE_CKPT: (out / 'best.pt').unlink()

if DO_B:
    # base/qaug first (mostly reloading checkpoints), then the new arms; all seeds of one model before the next
    todo = [(m, a, s) for m in MODELS for a in REF_ARMS for s in SEEDS] + [(m, a, s) for m in MODELS for s in SEEDS for a in NEW_ARMS]
    print(f'{len(todo)} runs: {[mshort(m) for m in MODELS]} x {ARMS} x seed {SEEDS}')
    for m, a, s in todo:
        train_one(m, a, s)
        if HAS_GPU: torch.cuda.empty_cache()


In [ ]:
# 5. Summary and verdict per architecture (rerunnable without a GPU)
def cluster_boot(d, grp, B):
    """d = per-image values (already averaged over seeds). Resample image groups -> (mean %, lower, upper)."""
    gs = pd.Series(np.asarray(d, float)).groupby(np.asarray(grp)).agg(['sum', 'count']).values
    rng = np.random.default_rng(0); idx = rng.integers(0, len(gs), (B, len(gs)))
    bs = 100 * gs[idx, 0].sum(1) / gs[idx, 1].sum(1)
    return dict(diff=100 * float(np.mean(d)), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5)), n_images=len(d), n_groups=len(gs))

def _probs(arm, s): return MDIR / f'{arm}__s{s}/test_probs.npz'
def _indicator(z, variant, metric):
    y, pred = z['y'], z[variant].argmax(1)
    if metric == 'other_as_disease':
        m = y == OTHER; return m, (pred[m] != OTHER).astype(float)
    m = y != OTHER; return m, (pred[m] == OTHER).astype(float)          # disease_as_other

def boot_diff(arm, ref, variant, metric='other_as_disease', B=N_BOOT):
    """(arm - ref) on test, paired by seed, cluster bootstrap over image groups (as in 13)."""
    seeds = [s for s in SEEDS if _probs(arm, s).exists() and _probs(ref, s).exists()]
    if not seeds: return None
    d = 0
    for s in seeds:
        m, ia = _indicator(np.load(_probs(arm, s)), variant, metric); _, ir = _indicator(np.load(_probs(ref, s)), variant, metric)
        d = d + (ia - ir)
    return dict(n_seeds=len(seeds), **cluster_boot(d / len(seeds), SPLIT['test']['group'].astype(str).values[m], B))

def boot_closure(arm, variant, B=N_BOOT):
    """CI of the share of the gap closed, (base - arm)/(base - qaug). Descriptive, not used for the verdict."""
    seeds = [s for s in SEEDS if all(_probs(a, s).exists() for a in ('base', 'qaug', arm))]
    if not seeds: return (np.nan, np.nan)
    e = {}
    for a in ('base', 'qaug', arm):
        e[a] = np.mean([_indicator(np.load(_probs(a, s)), variant, 'other_as_disease')[1] for s in seeds], 0)
    grp = SPLIT['test']['group'].astype(str).values[SPLIT['test']['y'].values == OTHER]
    g = pd.DataFrame(e).groupby(grp).sum().values                    # columns: base, qaug, arm
    rng = np.random.default_rng(0); idx = rng.integers(0, len(g), (B, len(g))); s_ = g[idx].sum(1)
    with np.errstate(divide='ignore', invalid='ignore'):
        cl = 100 * (s_[:, 0] - s_[:, 2]) / (s_[:, 0] - s_[:, 1])
    cl = cl[np.isfinite(cl)]
    return (float(np.percentile(cl, 2.5)), float(np.percentile(cl, 97.5))) if len(cl) else (np.nan, np.nan)


def load_results(mdir):
    rows = []
    for p in sorted(mdir.glob('*__s*/result.json')):
        j = json.load(open(p))
        for d in j['robustness']:
            rows.append(dict(model=j['model'], arm=j['arm'], seed=j['seed'], source=j.get('source'),
                             ref_ok=(j.get('ref_check') or {}).get('ok'), **d))
    return pd.DataFrame(rows)

LV = CRIT['level']
fmt = lambda x: 'no result' if x is None else ('met' if x else 'not met')
VERDICT = dict(criteria_sha=None, per_model={})
ALL = []
for m in MODELS:
    MDIR = OUT_DIR / mshort(m)
    res = load_results(MDIR) if MDIR.exists() else pd.DataFrame()
    print(f'\n################ {mshort(m)} ################')
    if not len(res):
        print('no results'); VERDICT['per_model'][mshort(m)] = dict(verdict='no results'); continue
    ALL.append(res)
    nseed = res.groupby('arm').seed.nunique().to_dict()
    src = res.groupby(['arm', 'seed']).source.first().unstack().to_string()
    print('result source (trained / reused_checkpoint):\n' + src)
    bad_ref = res[(res.source == 'reused_checkpoint') & (res.ref_ok == False)][['arm', 'seed']].drop_duplicates()
    if len(bad_ref): print('WARNING: reused checkpoints do not reproduce the stored results:', bad_ref.values.tolist())
    show = STD_LEVELS + ['jpeg_only', '112px_nojpeg', '80px_nojpeg']
    arms_done = [a for a in ARMS if a in res.arm.unique()]
    t = res.pivot_table(index='arm', columns='variant', values='other_as_disease', aggfunc=['mean', 'std'])
    tab = pd.DataFrame({v: [f'{t["mean"][v][a]:.1f}±{0 if np.isnan(t["std"][v][a]) else t["std"][v][a]:.1f}' for a in arms_done] for v in show if v in t['mean']}, index=arms_done)
    print(f'other->disease (%), mean±SD over seeds (n seeds = {nseed})'); print(tab.to_string())
    tab.to_csv(MDIR / 'table_other_as_disease.csv')
    mean = res.groupby(['arm', 'variant'])[['other_as_disease', 'disease_as_other', 'acc', 'acc_disease']].mean()
    rows = []
    for lv in ['112px', '80px']:
        if ('base', lv) not in mean.index or ('qaug', lv) not in mean.index: continue
        b = mean.loc[('base', lv), 'other_as_disease']; gap = b - mean.loc[('qaug', lv), 'other_as_disease']
        for a in arms_done:
            if a == 'base': continue
            bd = boot_diff(a, 'base', lv) or {}
            v = mean.loc[(a, lv), 'other_as_disease']
            clo, chi = boot_closure(a, lv) if a != 'qaug' else (np.nan, np.nan)
            rows.append(dict(model=mshort(m), level=lv, arm=a, base=b, arm_value=v, diff=bd.get('diff'), ci_lo=bd.get('lo'), ci_hi=bd.get('hi'),
                             gap=gap, gap_closed_pct=100 * (b - v) / gap if gap > 0 else np.nan, gap_closed_ci_lo=clo, gap_closed_ci_hi=chi, n_seeds=bd.get('n_seeds')))
    vt = pd.DataFrame(rows); vt.to_csv(MDIR / 'verdict_table.csv', index=False)
    if len(vt): print('\nDifference from base (arm - base, points, 95% cluster-bootstrap CI):'); print(vt.round(2).to_string(index=False))
    def get(a, col):
        r = vt[(vt.arm == a) & (vt.level == LV)] if len(vt) else vt
        return float(r[col].iloc[0]) if len(r) and pd.notna(r[col].iloc[0]) else None
    gap = get('qaug', 'gap'); have = all(nseed.get(a, 0) >= CRIT['G_min_seeds'] for a in ARMS)
    G = bool(have and gap is not None and gap >= CRIT['G_min_gap'])
    cj, hj, cr = get('qaug_jpeg', 'gap_closed_pct'), get('qaug_jpeg', 'ci_hi'), get('qaug_resize', 'gap_closed_pct')
    P1 = None if cj is None or hj is None else bool(cj >= CRIT['P1_jpeg_close_min'] and hj < 0)
    P2 = None if cr is None else bool(cr < CRIT['P2_resize_close_max'])
    print(f"\n===== Pre-specified criteria ({LV} + JPEG q{JPEG_Q}) =====")
    print(f"G: seeds per arm {nseed} | gap base - qaug = {None if gap is None else round(gap, 1)} pts (needs >= {CRIT['G_min_gap']}) -> " + ('met' if G else 'not met / incomplete'))
    if cj is not None: print(f"P1 qaug_jpeg closed {cj:.0f}% of the gap (needs >= {CRIT['P1_jpeg_close_min']:.0f}%) | upper CI of (qaug_jpeg - base) = {hj:.1f} (needs < 0) -> {fmt(P1)}")
    if cr is not None: print(f"P2 qaug_resize closed {cr:.0f}% of the gap (needs < {CRIT['P2_resize_close_max']:.0f}%) -> {fmt(P2)}")
    if not have: v_ = 'undecidable (not all arms/seeds complete)'
    elif not G: v_ = 'undecidable (base - qaug gap below criterion G)'
    elif None in (P1, P2): v_ = 'undecidable (incomplete)'
    elif P1 and P2: v_ = 'confirmed'
    else: v_ = 'not confirmed'
    print(f'>>> {mshort(m)}: {v_}')
    VERDICT['per_model'][mshort(m)] = dict(verdict=v_, G=G, P1=P1, P2=P2, gap=gap, closed_jpeg=cj, jpeg_ci_hi=hj, closed_resize=cr,
                                           closed_jpeg_ci=[get('qaug_jpeg', 'gap_closed_ci_lo'), get('qaug_jpeg', 'gap_closed_ci_hi')],
                                           closed_resize_ci=[get('qaug_resize', 'gap_closed_ci_lo'), get('qaug_resize', 'gap_closed_ci_hi')],
                                           sources=res.groupby(['arm', 'seed']).source.first().reset_index().values.tolist())

vs = [d['verdict'] for d in VERDICT['per_model'].values()]
n_ok = sum(v == 'confirmed' for v in vs)
VERDICT['overall'] = ('confirmed in both architectures: the JPEG result holds in three architectures (with B0 from 14)' if n_ok == len(MODELS) else
                      f'confirmed in {n_ok} of {len(MODELS)} architectures' if len(vs) == len(MODELS) and all(not v.startswith('undecidable') and v != 'no results' for v in vs) else
                      'no overall verdict (incomplete results or an undecidable architecture)')
VERDICT['criteria_sha'] = __import__('hashlib').sha256(json.dumps(CRIT, sort_keys=True).encode()).hexdigest()
print('\n========== Overall ==========\n' + VERDICT['overall'])
(OUT_DIR / 'verdict22.json').write_text(json.dumps(VERDICT, indent=1, ensure_ascii=False, default=str))
if ALL:
    allres = pd.concat(ALL); allres.to_csv(OUT_DIR / 'runs22.csv', index=False)
    # values for Figure 4 (mean and SD over seeds per standard level)
    fig4 = allres[allres.variant.isin(STD_LEVELS)].groupby(['model', 'arm', 'variant']).other_as_disease.agg(['mean', 'std', 'count']).reset_index()
    fig4.to_csv(OUT_DIR / 'fig4_values.csv', index=False)
    print('saved fig4_values.csv')


In [ ]:
# 6. Bundle results (without best.pt)
ZIP = OUT_DIR.parent / 'arch22_results.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT_DIR.rglob('*')):
        if p.is_file() and p.suffix != '.pt':
            z.write(p, Path('arch22') / p.relative_to(OUT_DIR))
print('saved', ZIP, f'({ZIP.stat().st_size/1e6:.1f} MB) | total {(time.time()-T_START)/60:.0f} min')
